# 0.2 Benchmarks

Where do commercial claims (Ecovative and its GIY kit, plus the companies in `MycoMaterials/mycomaterials_benchmarks.xlsx`) sit against the lab datasets in `data/interim/extracted/`?

**Status of each benchmark value** (column `status` below)
- `checked`: I read it myself on grow.bio or in the patent text.
- `patent`: Gemini found it in an uploaded patent and gave an exact quote (`MycoMaterials/Germini/germini_r1.xlsx` and `patent_benchmark_verification_notes.md`, 2026-10-11).
- `no_source`: no patent or primary source found. Kept in the table, left off the charts.

**Kind of value** (column `kind`): `measured` is a test result or worked example; `claim` is a claimed or preferred range written into the patent. Measured values are drawn solid, claims dashed.

Commercial figures are reference marks, never model training rows.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

from mycodemo.config import INTERIM_DATA_DIR

EXTRACTED_DIR = INTERIM_DATA_DIR / "extracted"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

# One colour per source family; each of our studies gets its own colour and marker shape
OURS, ECO, OTHER = "#2a78d6", "#eb6834", "#008300"
FAMILY_COLOUR = {"eco": ECO, "other": OTHER}
DATASET_STYLE = {  # colour, marker
    "d004": ("#2a78d6", "o"),
    "d005": ("#1baf7a", "s"),
    "d007": ("#6250d6", "^"),
    "d016": ("#e87ba4", "v"),
}
NAMES = {
    "d002": "Musci 2025", "d003": "Gomaa 2024", "d004": "Bhaskar 2026", "d005": "Elsacker 2019",
    "d006": "Hysek 2025", "d007": "Irbe 2026", "d009": "Elsacker 2022", "d016": "Vašatko 2022",
    "d012": "Pacheco-Sanchez 2026", "d017": "Jiménez-Obando 2025",
    "d023": "Houette 2022",
}

## 1. Benchmark values

Typed in here rather than kept as a new CSV. One row per value, so a product can have several rows. Units: density kg/m³, stress and tensile MPa, days, °C. `lo` = `hi` means a single value; an empty `hi` means the source gives only a lower limit (">"). GIY conversions: 7.6 lb/ft³ ≈ 122 kg/m³, 18 psi ≈ 0.124 MPa.

In [ ]:
cols = ["benchmark", "maker", "family", "property", "lo", "hi", "kind", "status", "source", "note"]
rows = [
    # Ecovative and its GIY kit
    ["GIY kit material", "Ecovative", "eco", "density", 122, 122, "measured", "checked", "grow.bio product page", "maker's spec sheet value, no replicates"],
    ["GIY kit material", "Ecovative", "eco", "stress", 0.124, 0.124, "measured", "checked", "grow.bio product page", "at 15% strain; flexural 0.23 MPa"],
    ["GIY kit", "Ecovative", "eco", "days", 7, 10, "measured", "checked", "grow.bio FAQ and instructions", "thin parts, pre-colonised material"],
    ["GIY kit", "Ecovative", "eco", "temp", 22, 22, "measured", "checked", "grow.bio instructions", ""],
    ["Grown.bio kit (EU licensee)", "Ecovative", "eco", "days", 6, 7, "measured", "checked", "Grown.bio kit manual", ""],
    ["Grown.bio kit (EU licensee)", "Ecovative", "eco", "temp", 22, 26, "measured", "checked", "Grown.bio kit manual", ""],
    ["Ecovative grown materials", "Ecovative", "eco", "days", 14, 14, "measured", "checked", "US9485917B2, Examples 4 and 6", "P. ostreatus"],
    ["Ecovative grown materials", "Ecovative", "eco", "temp", 24, 24, "measured", "checked", "US9485917B2, Examples 4 and 6", "75 °F, 100% RH"],
    ["Ecovative moulded packaging", "Ecovative", "eco", "days", 3, 12, "claim", "patent", "US11343979B2, operating parameters table", "worked example 5 + 4 = 9 days"],
    ["Ecovative moulded packaging", "Ecovative", "eco", "temp", 18, 24, "measured", "patent", "US11343979B2, operating parameters table", "65 °F cooled air, 75 °F air; vessel below 35 °C"],
    ["Ecovative moulded packaging", "Ecovative", "eco", "density", 50, 250, "claim", "no_source", "review literature per Gemini (Jones 2017, Girometta 2019)", "not in US11343979B2"],
    ["Ecovative moulded packaging", "Ecovative", "eco", "stress", 0.2, 1.5, "claim", "no_source", "review literature per Gemini", "not in US11343979B2"],
    ["Ecovative moulded packaging", "Ecovative", "eco", "thermal", 0.040, 0.040, "claim", "no_source", "review literature per Gemini", "W/m·K"],
    ["Ecovative AirMycelium", "Ecovative", "eco", "density", 10, 800, "claim", "no_source", "Elsacker 2023 review per Gemini", "patents not uploaded"],
    ["Ecovative AirMycelium", "Ecovative", "eco", "days", 9, 14, "claim", "no_source", "Elsacker 2023 review per Gemini", ""],
    ["Ecovative AirMycelium", "Ecovative", "eco", "temp", 30, 32, "claim", "no_source", "Elsacker 2023 review per Gemini", ""],
    ["Ecovative fruiting-body shapes", "Ecovative", "eco", "days", 28, 75, "measured", "patent", "US8001719B2, Examples 1 and 2", "21–45 d incubation + 7–30 d fruiting"],
    ["Ecovative fruiting-body shapes", "Ecovative", "eco", "temp", 21, 24, "measured", "patent", "US8001719B2, Examples 1 and 2", ""],
    ["Raw fruiting-body tissue", "Ecovative", "eco", "density", 180, 390, "measured", "patent", "US8001719B2, col. 1", "cap 180, stipe up to 350, spore tubes 340–390; tissue, not composite"],
    # Other companies
    ["Okom Wrks boards", "Okom Wrks", "other", "density", 250, None, "claim", "patent", "US11866691B2, claims 1 and 19", "> 0.25 g/cm³"],
    ["Okom Wrks boards", "Okom Wrks", "other", "stress", 1.2, None, "claim", "patent", "US11866691B2, claims 1 and 19", "> 1.2 MPa ultimate compressive strength"],
    ["Okom Wrks example", "Okom Wrks", "other", "density", 250, 250, "measured", "patent", "US11866691B2, Example 1", "hollow cylinder, n = 1"],
    ["Okom Wrks example", "Okom Wrks", "other", "stress", 0.98, 0.98, "measured", "patent", "US11866691B2, Example 1", "ultimate compressive strength, strain not given; below its own 1.2 MPa claim"],
    ["Okom Wrks boards", "Okom Wrks", "other", "days", 4, 17, "claim", "patent", "US11866691B2, claims 2 and 12–14", ""],
    ["Okom Wrks boards", "Okom Wrks", "other", "temp", 22, 26, "claim", "patent", "US11866691B2, claims", ""],
    ["Processed mycelium leather", "assignee to confirm", "other", "density", 620, 621, "measured", "patent", "US20220007777A1, Table 1", "G. sessile mats, n = 146 and 151"],
    ["Processed mycelium leather", "assignee to confirm", "other", "tensile", 2.87, 3.69, "measured", "patent", "US20220007777A1, Table 1", "means of treatments A and B (SD 1.3–1.4), n = 42 and 47"],
    ["SQIM ephea, uncoated", "SQIM / MOGU", "other", "tensile", 0.5, 5, "claim", "patent", "WO2021245608A1, p. 6", "ISO 3376"],
    ["SQIM ephea, coated", "SQIM / MOGU", "other", "tensile", 8, 50, "claim", "patent", "WO2021245608A1, p. 8", "coating about 8× stronger in Example 4"],
    ["SQIM ephea", "SQIM / MOGU", "other", "days", 10, 18, "claim", "patent", "WO2021245608A1, p. 5", "preferred mat incubation; Example 2 used 14 d"],
    ["SQIM ephea", "SQIM / MOGU", "other", "temp", 23, 28, "measured", "patent", "WO2021245608A1, Example 2", ""],
    ["Bolt Threads Mylo", "Bolt Threads", "other", "tensile", 5, 30, "claim", "patent", "WO2022226322A1, para. 0120", "wet tensile strength"],
    ["Bolt Threads Mylo", "Bolt Threads", "other", "temp", 20, 35, "claim", "patent", "WO2022226322A1, para. 0148", "growth days not stated"],
    ["Ford plasticised composites", "Ford", "other", "days", 1, 15, "claim", "patent", "US8227225B2, col. 7", ""],
    ["Ford plasticised composites", "Ford", "other", "temp", 13, 32, "claim", "patent", "US8227225B2, col. 4", "55–90 °F"],
    ["MycoWorks Fine Mycelium", "MycoWorks", "other", "tensile", 10, 25, "claim", "no_source", "none found", ""],
    ["MycoWorks Fine Mycelium", "MycoWorks", "other", "days", 14, 18, "claim", "no_source", "Elsacker 2023 review per Gemini", ""],
    ["Mushroom Material pellets", "Mushroom Material", "other", "density", 40, 120, "claim", "no_source", "news release per Gemini", ""],
    ["Mushroom Material pellets", "Mushroom Material", "other", "days", 6, 8, "claim", "no_source", "news release per Gemini", ""],
    ["Biohm MycoPanel", "Biohm", "other", "thermal", 0.024, 0.024, "claim", "no_source", "marketing per Gemini", "below still air (~0.026): likely wrong"],
    ["Mykor MykoFoam", "Mykor", "other", "thermal", 0.030, 0.030, "claim", "no_source", "Innovate UK case study per Gemini", ""],
    ["Loop Living Cocoon", "Loop Biotech", "other", "days", 7, 7, "claim", "no_source", "product page per Gemini", ""],
]
bench = pd.DataFrame(rows, columns=cols)
shown = bench[bench.status != "no_source"]  # charts use only checked and patent values

print(bench.status.value_counts().to_string())
bench

## 2. Our data: density and stress per row, growth days and temperature per dataset

In [ ]:
meas = pd.read_csv(EXTRACTED_DIR / "measurement.csv")
sample = pd.read_csv(EXTRACTED_DIR / "sample.csv")
ferm = pd.read_csv(EXTRACTED_DIR / "fermentation.csv")

props = meas.rename(columns={"compressive_stress_at_10pct_strain_MPa": "stress",
                             "dry_density_kg_m3": "density"})
both = props.dropna(subset=["stress", "density"])
dens = props.dropna(subset=["density"])

growth = (sample.merge(ferm, on="fermentation_id")
          .groupby("dataset_id")[["total_growth_days", "incubation_temp_c"]]
          .agg(["min", "max"]))
growth.columns = ["days_lo", "days_hi", "temp_lo", "temp_hi"]

print(f"{len(both)} rows have both density and stress, from {both.dataset_id.nunique()} datasets")
print(f"{len(dens)} rows have density, from {dens.dataset_id.nunique()} datasets")
growth

## 3. Density against compressive strength, coloured by study

Both axes are logarithmic. Our values are stress at 10% strain. GIY is at 15% strain, which reads a little higher. The Okom example is an ultimate compressive strength with no strain given. Hollow markers are group means, filled markers single blocks.

In [ ]:
def pick(name, prop):
    r = shown[(shown.benchmark == name) & (shown.property == prop)]
    return r.iloc[0] if len(r) else None

fig, ax = plt.subplots(figsize=(9, 6))
ax.set(xscale="log", yscale="log", xlim=(20, 1000), ylim=(0.002, 3),
       xlabel="Dry density (kg/m³)", ylabel="Compressive stress (MPa)")
ax.grid(True, which="major", color="#e6e8e3", lw=0.8)
xt, yt = [20, 50, 100, 200, 500, 1000], [0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5, 1, 2]
ax.set_xticks(xt, [str(v) for v in xt])
ax.set_yticks(yt, [str(v) for v in yt])
ax.minorticks_off()

# Okom claimed region (lower limits only, so the box runs to the chart edge)
d, s = pick("Okom Wrks boards", "density"), pick("Okom Wrks boards", "stress")
ax.add_patch(Rectangle((d.lo, s.lo), 1000 - d.lo, 3 - s.lo, facecolor=OTHER, alpha=0.08,
                       edgecolor=OTHER, ls="--", lw=1.5))
ax.text(d.lo * 1.04, 2.6, "Okom Wrks claim (≥)", color=OTHER, fontsize=9, va="top")

# Our studies
for ds, (colour, marker) in DATASET_STYLE.items():
    d = both[both.dataset_id == ds]
    raw, mean = d[d.measurement_type == "raw"], d[d.measurement_type == "mean"]
    ax.scatter(raw.density, raw.stress, c=colour, marker=marker, s=28, alpha=0.8,
               edgecolors="white", linewidths=0.5)
    ax.scatter(mean.density, mean.stress, facecolors="none", edgecolors=colour,
               marker=marker, s=42, linewidths=1.3)

# Single measured benchmark points
points = [("GIY kit material", ECO, "D", "GIY kit (15% strain)", (-10, 10), "right"),
          ("Okom Wrks example", OTHER, "P", "Okom example (n = 1)", (10, -4), "left")]
for name, colour, marker, label, offset, ha in points:
    x, y = pick(name, "density").lo, pick(name, "stress").lo
    ax.scatter([x], [y], marker=marker, s=110, c=colour, edgecolors="white", linewidths=1.5, zorder=5)
    ax.annotate(label, (x, y), xytext=offset, textcoords="offset points", ha=ha,
                color=colour, fontsize=9)

handles = [Line2D([], [], ls="", marker=m, color=c, label=f"{NAMES[ds]} (n={len(both[both.dataset_id == ds])})")
           for ds, (c, m) in DATASET_STYLE.items()]
handles += [Line2D([], [], ls="", marker="D", color=ECO, label="GIY kit (checked)"),
            Line2D([], [], ls="", marker="P", color=OTHER, label="Okom patent example"),
            Patch(facecolor="none", edgecolor="grey", ls="--", label="claimed range (patent)"),
            Line2D([], [], ls="", marker="o", mfc="none", color="grey", label="hollow = group mean")]
ax.legend(handles=handles, loc="lower right", fontsize=8.5, frameon=False)
ax.set_title("Density against compressive strength", loc="left", fontweight="bold")
plt.show()

**What it shows:** the two sourced commercial points sit at the top of our cloud, not above it. The GIY kit (122 kg/m³, 0.12 MPa) lands among the lab blocks. The Okom example (250 kg/m³, 0.98 MPa) is stronger than any lab block at that density, but it falls short of its own patent claim of 1.2 MPa. Ecovative's moulded-packaging box from the first version is gone because no patent gives those numbers. Within each colour, density predicts strength only weakly.

In [ ]:
def range_rows(ax, items, xmax, invert=True):
    """One horizontal bar (or dot) per item; items are (label, lo, hi, colour, dashed)."""
    for i, (label, lo, hi, colour, dashed) in enumerate(items):
        if pd.isna(lo):
            continue
        open_hi = pd.isna(hi)
        hi = xmax if open_hi else hi
        if lo == hi:
            ax.scatter([lo], [i], c=colour, s=45, zorder=3, edgecolors="white")
        else:
            ax.barh(i, hi - lo, left=lo, height=0.5,
                    color=colour, alpha=0.15 if dashed else 0.85,
                    edgecolor=colour, ls="--" if dashed else "-", lw=1.3)
        if open_hi:
            ax.annotate("", (xmax, i), (xmax * 0.9 if ax.get_xscale() == "log" else xmax - 1, i),
                        arrowprops=dict(arrowstyle="->", color=colour))
    ax.set_yticks(range(len(items)), [it[0] for it in items])
    if invert:  # only once per shared y-axis
        ax.invert_yaxis()
    ax.grid(True, axis="x", color="#e6e8e3", lw=0.8)


def bench_items(prop):
    """Benchmark rows for one property, as range_rows items (claims dashed)."""
    b = shown[shown.property == prop]
    return [(r.benchmark, r.lo, r.hi, FAMILY_COLOUR[r.family], r.kind == "claim")
            for r in b.itertuples()]

## 4. Density of every source

In [ ]:
items, dots = [], []
for ds in ["d004", "d005", "d006", "d007", "d016", "d017", "d023"]:
    v = dens.loc[dens.dataset_id == ds, "density"]
    items.append((NAMES[ds], v.min(), v.max(), OURS, False))
    dots.append(v.values)
items += bench_items("density")

fig, ax = plt.subplots(figsize=(9, 5))
ax.set_xscale("log")
ax.set_xlim(20, 1000)
ax.set_xticks([20, 50, 100, 200, 500, 1000], ["20", "50", "100", "200", "500", "1000"])
ax.minorticks_off()
range_rows(ax, items, 1000)
for i, v in enumerate(dots):  # individual rows on top of our bars
    ax.scatter(v, np.full(len(v), i) + np.random.default_rng(i).uniform(-0.15, 0.15, len(v)),
               s=6, c="white", zorder=4)
ax.set_xlabel("Dry density (kg/m³, log scale)")
ax.set_title("Density by source", loc="left", fontweight="bold")
plt.show()

**What it shows:** lab blocks run from about 30 to 280 kg/m³. GIY sits in the middle of that band and Okom at its top. Pressed leather mats (about 620 kg/m³) and raw fruiting-body tissue (180–390 kg/m³) are different materials, shown for scale.

## 5. Growth time and temperature

A missing bar means the source gives no value. The syldd batch is left out until it finishes.

In [ ]:
items_d, items_t = [], []
for ds, g in growth.iterrows():
    items_d.append((NAMES.get(ds, ds), g.days_lo, g.days_hi, OURS, False))
    items_t.append((NAMES.get(ds, ds), g.temp_lo, g.temp_hi, OURS, False))
names = list(dict.fromkeys(shown[shown.property.isin(["days", "temp"])].benchmark))
for name in names:
    for prop, items in [("days", items_d), ("temp", items_t)]:
        r = shown[(shown.benchmark == name) & (shown.property == prop)]
        if len(r):
            r = r.iloc[0]
            items.append((name, r.lo, r.hi, FAMILY_COLOUR[r.family], r.kind == "claim"))
        else:
            items.append((name, np.nan, np.nan, OURS, False))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 8), sharey=True,
                             gridspec_kw={"width_ratios": [1.3, 1]})
a1.set_xlim(0, 80)
a2.set_xlim(10, 36)
range_rows(a1, items_d, 80)
range_rows(a2, items_t, 36, invert=False)
a1.set_xlabel("Growth cycle (days)")
a2.set_xlabel("Incubation temperature (°C)")
fig.suptitle("Growth time and temperature by source", x=0.02, ha="left", fontweight="bold")
fig.tight_layout()
plt.show()

**What it shows:** the kits and the packaging patent finish in 3–14 days at 18–26 °C. Most lab studies took 14–29 days, and Houette 2022 took 46. Kits use thin parts and pre-colonised material, so their days are not a fair target for M1a in a 3D mould. Ecovative's fruiting-body shapes (28–75 days) are the one commercial route slower than the lab.

## 6. Tensile strength of mycelium leathers

None of our studies measures tensile strength, so these stand alone for now.

In [ ]:
items = bench_items("tensile")
fig, ax = plt.subplots(figsize=(9, 2.8))
ax.set_xlim(0, 52)
range_rows(ax, items, 52)
ax.set_xlabel("Tensile strength (MPa)")
ax.set_title("Mycelium leather tensile strength (patents)", loc="left", fontweight="bold")
plt.show()

**What it shows:** the only measured leather values (2.9 and 3.7 MPa, US20220007777A1) sit at the low end of every claimed range. The claims run up to 50 MPa for coated SQIM material. Claims in patents are what the applicant wants to protect, not what they achieved.

## 7. Caveats

- Commercial values are reference marks without replicates (except the leather table), never training rows.
- Strain differs: ours 10%, GIY 15%, Okom not stated.
- Kit and factory cycle days use thin parts and pre-colonised material, so they are not a fair target for M1a in a 3D mould.
- `no_source` rows (Ecovative packaging density and strength, AirMycelium, MycoWorks, Mushroom Material, Biohm, Mykor, Loop) stay in the table only. Biohm's 0.024 W/m·K is below still air and is probably wrong.
- Gemini credits US20220007777A1 to MycoWorks; the Google Patents record lists BLC Leather Technology Centre and Bolt Threads. The assignee is to be confirmed before citing.